In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

DB_USER = 'root'          # MySQL username
DB_PASSWORD = input('Password')  # MySQL password
DB_HOST = 'localhost'
DB_NAME = 'financial_analytics'

# Connection engine
engine = create_engine(f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}')

# Test connection
try:
    with engine.connect() as conn:
        print("✅ Successfully connected to MySQL!")
except Exception as e:
    print(f" Connection failed: {e}")
    exit()


# Update these file paths to match folder structure
customers = pd.read_csv('Customers.csv')
vendors = pd.read_csv('Vendors.csv')
headcount = pd.read_csv('Headcount.csv')
budget = pd.read_csv('Budget.csv')
transactions = pd.read_csv('Financial_Transactions.csv')

print(f"   Data loaded:")
print(f"   Customers: {len(customers)} rows")
print(f"   Vendors: {len(vendors)} rows")
print(f"   Headcount: {len(headcount)} rows")
print(f"   Budget: {len(budget)} rows")
print(f"   Transactions: {len(transactions)} rows")

# Data Cleaning


# Convert Date Columns
customers['join_date'] = pd.to_datetime(customers['join_date']).dt.date
headcount['join_date'] = pd.to_datetime(headcount['join_date']).dt.date
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date']).dt.date


# Replace NaN/empty strings with None (becomes NULL in SQL)
transactions['customer_id'] = transactions['customer_id'].replace('', None)
transactions['vendor_id'] = transactions['vendor_id'].replace('', None)

# Ensure amount is float
transactions['amount'] = transactions['amount'].astype(float)

# Ensure budget and CTC are integers
budget['budgeted_revenue'] = budget['budgeted_revenue'].astype(int)
budget['budgeted_expense'] = budget['budgeted_expense'].astype(int)
headcount['cost_to_company'] = headcount['cost_to_company'].astype(int)

print("Data cleaning complete!")


# Insert into MySQL

try:
    print("Inserting Customers...")
    customers.to_sql('customers', engine, if_exists='append', index=False)

    print("Inserting Vendors...")
    vendors.to_sql('vendors', engine, if_exists='append', index=False)

    print("Inserting Headcount...")
    headcount.to_sql('headcount', engine, if_exists='append', index=False)

    print(" Inserting Budget...")
    budget.to_sql('budget', engine, if_exists='append', index=False)

    print("Inserting Financial Transactions (110k rows, this may take 1-2 minutes)...")
    transactions.to_sql('financial_transactions', engine, 
                           if_exists='append', index=False, chunksize=5000)

    print("ALL DATA SUCCESSFULLY LOADED!")

except Exception as e:
    print(f"Error during insertion: {e}")


# Verify Row Counts

with engine.connect() as conn:
    print("\n Verification - Row Counts:")
    tables = ['customers', 'vendors', 'headcount', 'budget', 'financial_transactions']
    for i in tables:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {i}"))
        count = result.scalar()
        print(f"   {i}: {count:,} rows")

Password 671161


✅ Successfully connected to MySQL!
   Data loaded:
   Customers: 400 rows
   Vendors: 120 rows
   Headcount: 200 rows
   Budget: 72 rows
   Transactions: 10400 rows
Data cleaning complete!
Inserting Customers...
Error during insertion: (pymysql.err.IntegrityError) (1062, "Duplicate entry 'CUST10000' for key 'customers.PRIMARY'")
[SQL: INSERT INTO customers (customer_id, customer_name, segment, join_date, region, status) VALUES (%(customer_id)s, %(customer_name)s, %(segment)s, %(join_date)s, %(region)s, %(status)s)]
[parameters: [{'customer_id': 'CUST10000', 'customer_name': 'Johnson, Olson and Smith', 'segment': 'Online', 'join_date': datetime.date(2020, 3, 28), 'region': 'North', 'status': 'Active'}, {'customer_id': 'CUST10001', 'customer_name': 'Colon-Valdez', 'segment': 'Enterprise', 'join_date': datetime.date(2018, 2, 15), 'region': 'West', 'status': 'Active'}, {'customer_id': 'CUST10002', 'customer_name': 'Henry, Jones and Farley', 'segment': 'Online', 'join_date': datetime.date(2